# P1f: preserve transferable disease features, then evaluate P3

P1 and P2 have no confirmed final winner. ConvNeXt-Tiny remains the chosen backbone.
P2e's cross-crop/adversary setup gave 24.33% unseen disease Top-1 versus 21.53% for its control,
below the agreed comparison threshold. P3 prototypes on those features did not improve disease accuracy.

This experiment tests three classification recipes with seeds 42, 43 and 44, ten epochs each.

| Config | Backbone | Backbone learning rate |
|---|---|---|
| `p1f_ctl` | Fully trainable ConvNeXt-Tiny | `1e-4` |
| `p1f_low_lr` | Fully trainable ConvNeXt-Tiny | `3e-5` |
| `p1f_frozen` | Frozen pretrained ConvNeXt-Tiny | No backbone updates |

All three use identical heads, head learning rate `1e-3`, augmentation, training splits, two views per image,
and 4,310 applied updates. The frozen backbone stays in inference mode. Its normalization statistics and
stochastic layers remain fixed while the heads train. Patch shuffle, contrastive loss and the crop adversary are off.
These are new runs, not fine-tuning of the old P2e checkpoints.

P3 runs automatically on epoch ten of every completed run. It compares the classifier with pooled and
crop-balanced prototypes and reports direct unseen disease Top-1. P3 is a final-epoch diagnostic, while the
training comparison uses each run's best full three-epoch window. Do not compare these two score windows as equals.

Keep `SMOKE = True` for the first Run All. Three two-epoch smoke runs should finish without errors.
The frozen run must print `trainable parameters: backbone=0` and a positive heads count.
Then change `SMOKE = False` and use Save Version with Save & Run All.

Attach one code dataset containing `cdc-cp_code_p1f.zip` and the existing DPD-Small data dataset.
Select GPU T4 x2 and enable Internet. Do not attach the old code dataset alongside the new one.
Training happens on Kaggle. Only the original training CSV and the development split images are used.
Official test files are never opened.

If a full run is interrupted, add that P1f version's output as an input and run again.
Only compatible P1f checkpoint recipes are restored. Smoke outputs are skipped.

Results are saved in `/kaggle/working/p1f_runs/`.
Read `p1f_compare.md`, `p1f_runs.csv`, `p1f_p3_runs.csv` and each run's `p3_board_ep10.md`.
The disease verdict uses paired seeds, 400 unseen images and `z=2.4`, retained conservatively for two comparisons.
Review seen disease accuracy and pair diagnostics before adopting any recipe. There is no automatic final-model selection.

## 1. Settings

In [ ]:
from pathlib import Path

SMOKE = True
TOL = 3.0
WORKERS = 2
Z = 2.4
CONTROL = 'p1f_ctl'
CONFIGS = [CONTROL, 'p1f_low_lr', 'p1f_frozen']
SEEDS = (42, 43, 44)
INPUT = Path('/kaggle/input')
RUNS_DIR = Path('/kaggle/working/p1f_smoke' if SMOKE else '/kaggle/working/p1f_runs')
CODE_DIR = Path('/tmp/cdc-cp')
DATA_DIR = Path('/tmp/DPD_Small')
CACHE_DIR = Path('/tmp/cache')

## 2. GPUs and code

In [ ]:
import hashlib, os, shutil, subprocess, sys, zipfile, time
import torch

assert torch.cuda.is_available(), 'Settings > Accelerator > GPU T4 x2'
N_GPU = torch.cuda.device_count()
print(N_GPU, 'x', torch.cuda.get_device_name(0), '| torch', torch.__version__)
if N_GPU < 2:
    print('Only one GPU: runs will go one at a time (about twice as long). Pick GPU T4 x2 to run two at once.')

os.chdir('/kaggle/working')  # a rerun deletes CODE_DIR, so the kernel must not be inside it
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
def is_p1f_code(root):
    # a code folder is usable when it has every P1f feature this notebook needs
    needs = {'cdccp/models.py': 'freeze_backbone', 'scripts/p3_eval.py': '"unseen disease"', 'cdccp/evaluate.py': 'def pair_breakdown'}
    return all((root / f).is_file() and text in (root / f).read_text(encoding='utf-8') for f, text in needs.items())


def code_digest(root):
    # what training and scoring run: the package, the scripts and the configs (tests and notebooks may differ)
    files = sorted(p for d in ('cdccp', 'scripts', 'configs') for p in (root / d).rglob('*') if p.suffix in ('.py', '.yaml'))
    return hashlib.sha256(b''.join(p.relative_to(root).as_posix().encode() + p.read_bytes() for p in files)).hexdigest()


def pick_code(input_dir, unzip_to):
    """The one P1f code folder among the inputs. Old code folders (an older dataset version left beside the new zip,
    or another attached dataset) are skipped. P1f folders whose training code differs are refused, because the run
    would be ambiguous. Zips Kaggle did not unpack are unpacked first, into a fresh unzip_to."""
    if unzip_to.exists():
        shutil.rmtree(unzip_to)
    unzip_to.mkdir(parents=True)
    for i, z in enumerate(sorted(input_dir.rglob('cdc-cp_code_p1f*.zip'))):
        with zipfile.ZipFile(z) as archive:
            archive.extractall(unzip_to / str(i))
    folders = sorted({p.parent.parent for d in (input_dir, unzip_to) for p in d.rglob('cdccp/__init__.py')})
    good = [f for f in folders if is_p1f_code(f)]
    for f in folders:
        print('code folder:', f, '(P1f)' if f in good else '(old code, skipped)')
    assert good, f'No P1f code among the inputs {folders}: upload cdc-cp_code_p1f.zip as a new version of the code dataset'
    digests = {code_digest(f) for f in good}
    assert len(digests) == 1, f'{len(digests)} different P1f code folders {good}: keep only the newest cdc-cp_code_p1f.zip in the dataset'
    return good[0]


found = pick_code(INPUT, Path('/tmp/code_zips'))
shutil.copytree(found, CODE_DIR)
assert is_p1f_code(CODE_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml'], check=True)
os.chdir(CODE_DIR)
sys.path.insert(0, str(CODE_DIR))
missing_cfg = [c for c in CONFIGS if not (CODE_DIR / 'configs' / f'{c}.yaml').is_file()]
assert not missing_cfg, f'code dataset lacks configs {missing_cfg}: upload the new code zip'
from cdccp.config import load_config
from cdccp.evaluate import pair_breakdown
from cdccp.selection import compare_arms, run_scores
print('code from', found)


def sh(*args):
    proc = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    if proc.wait():
        raise RuntimeError(f'exit code {proc.returncode}: {args}')

## 3. Training data (train CSV and its 8,000 images only)

In [ ]:
import hashlib
import pandas as pd

train_csv = next(p for p in INPUT.rglob('*') if p.name.lower() == 'dpd_small_train.csv')
cols = ['filename', 'crop_id', 'disease_id', 'composition_id', 'crop_name', 'disease_name', 'composition_name']
train = pd.read_csv(train_csv, header=None, names=cols)
assert len(train) == 8000, f'expected 8,000 rows, got {len(train)}'
digest = hashlib.sha256(train_csv.read_bytes()).hexdigest()
assert digest.startswith('9f1b729a'), f'train CSV hash {digest[:8]} does not match the split manifest'

first = train.filename.iloc[0]
image_src = next((d for d in [train_csv.parent, *train_csv.parent.parent.rglob('*')] if d.is_dir() and (d / first).is_file()), None)
assert image_src is not None, f'could not find {first} near {train_csv.parent}'
DATA_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(train_csv, DATA_DIR / 'DPD_Small_train.csv')
link = DATA_DIR / 'DPD_Small_dataset'
if not link.exists():
    link.symlink_to(image_src, target_is_directory=True)
missing = [n for n in train.filename if not (link / n).is_file()]
assert not missing, f'{len(missing)} images missing, e.g. {missing[:3]}'
os.environ['DPD_ROOT'] = str(DATA_DIR)
print(f'8,000 training images linked from {image_src}')

## 4. Restore runs from earlier outputs

In [ ]:
import numpy as np
import yaml

RUNS_DIR.mkdir(parents=True, exist_ok=True)


def run_name(config, seed):
    return f'{config}_seed{seed}'


def planned_epochs(config):
    return load_config(CODE_DIR / 'configs' / f'{config}.yaml')['epochs']


def expected_recipe(config, seed):
    cfg = load_config(CODE_DIR / 'configs' / f'{config}.yaml', [f'seed={seed}'])
    return {k: v for k, v in cfg.items() if k not in ('paths', 'loader')}


def checkpoint_state(run_dir):
    return torch.load(run_dir / 'last.pt', map_location='cpu', weights_only=False)


def applied(run_dir):
    return checkpoint_state(run_dir)['applied']


def finished(config, seed):
    run_dir = RUNS_DIR / run_name(config, seed)
    n = planned_epochs(config)
    required = ['history.csv', 'last.pt', f'val_{n:02d}.npz', f'epoch_{n:02d}.pt']
    if not all((run_dir / p).is_file() for p in required):
        return False
    h = pd.read_csv(run_dir / 'history.csv')
    if list(h.epoch) != list(range(1, n + 1)):
        return False
    cfg = load_config(CODE_DIR / 'configs' / f'{config}.yaml')
    fit = pd.read_csv(Path(cfg['paths']['splits_dir']) / 'train_fit.csv')
    steps = len(fit) // cfg['batch_size']
    if list(h.applied) != [steps * e for e in range(1, n + 1)]:
        return False
    state = checkpoint_state(run_dir)
    return state.get('recipe') == expected_recipe(config, seed) and state['applied'] == steps * n


restored = []
if not SMOKE:
    for config in CONFIGS:
        for seed in SEEDS:
            dst = RUNS_DIR / run_name(config, seed)
            for last in INPUT.rglob(f'{run_name(config, seed)}/last.pt'):
                if 'p1f_smoke' in last.parts:
                    continue
                state = checkpoint_state(last.parent)
                if state.get('recipe') != expected_recipe(config, seed):
                    print('Skipped incompatible checkpoint:', last)
                    continue
                if not (dst / 'last.pt').exists() or applied(dst) < state['applied']:
                    shutil.rmtree(dst, ignore_errors=True)
                    shutil.copytree(last.parent, dst)
                    restored.append(str(last.parent))
print('restored:', restored or 'nothing')

## 5. Image cache (local disk, a few minutes)

In [ ]:
sh(sys.executable, 'scripts/build_cache.py', '--verify', '--workers', 8, f'paths.cache_dir={CACHE_DIR}')

## 6. Run queue
Each GPU takes the next waiting run. Only the last 3 epoch checkpoints of a run are kept.

In [ ]:
COMMON = [f'paths.runs_dir={RUNS_DIR}', f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']


def job(config, seed=42):
    args = [sys.executable, 'scripts/train.py', f'configs/{config}.yaml', *COMMON, f'seed={seed}', '--resume']
    if SMOKE:
        args += ['--max-epochs', '2']
    return run_name(config, seed), args


def prune(run_dir, keep=3):
    for p in sorted(run_dir.glob('epoch_*.pt'))[:-keep]:
        p.unlink()


def run_queue(jobs):
    """Run (name, args) jobs, one per GPU at a time. A failed run is reported and left out of the ranking."""
    waiting, running, failed = list(jobs), {}, []
    print('queue:', [n for n, _ in waiting], flush=True)
    while waiting or running:
        for gpu in range(max(N_GPU, 1)):
            if gpu not in running and waiting:
                name, args = waiting.pop(0)
                log = open(RUNS_DIR / f'{name}.log', 'a')
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
                p = subprocess.Popen([str(a) for a in args], cwd=CODE_DIR, env=env, stdout=log, stderr=subprocess.STDOUT, text=True)
                running[gpu] = (name, p, log, 0, time.time())
                print(f'[gpu {gpu}] start {name}', flush=True)
        time.sleep(20)
        for gpu, (name, p, log, seen, t0) in list(running.items()):
            log.flush()
            lines = (RUNS_DIR / f'{name}.log').read_text(errors='replace').splitlines()
            for line in lines[seen:]:
                if line.startswith(('epoch', 'done', 'nothing to do', 'Traceback')) or 'Error' in line:
                    print(f'[gpu {gpu}] {name}: {line}', flush=True)
            running[gpu] = (name, p, log, len(lines), t0)
            if p.poll() is not None:
                log.close()
                del running[gpu]
                mins = (time.time() - t0) / 60
                if p.returncode:
                    failed.append(name)
                    print(f'[gpu {gpu}] FAILED {name} after {mins:.0f} min; last lines:', *lines[-15:], sep='\n', flush=True)
                else:
                    print(f'[gpu {gpu}] finished {name} in {mins:.0f} min', flush=True)
                    if (RUNS_DIR / name).is_dir():
                        prune(RUNS_DIR / name)
    if failed:
        print(f'WARNING failed runs, left out of the ranking: {failed} (logs in {RUNS_DIR})', flush=True)
    return failed


def p3_job(config, seed):
    run_dir = RUNS_DIR / run_name(config, seed)
    return f'p3_{run_name(config, seed)}', [sys.executable, 'scripts/p3_eval.py', run_dir, '--epoch', planned_epochs(config),
                                            '--tol', TOL, f'paths.cache_dir={CACHE_DIR}', f'loader.num_workers={WORKERS}']

## 7. Train three feature-learning recipes across three seeds

In [ ]:
FAILED = run_queue([job(c, s) for s in ((42,) if SMOKE else SEEDS) for c in CONFIGS])
if SMOKE:
    import re
    assert not FAILED, f'Smoke runs failed: {FAILED}. Read the failed logs before a full run.'
    frozen_log = (RUNS_DIR / 'p1f_frozen_seed42.log').read_text()
    counts = re.findall(r'trainable parameters: backbone=(\d+) heads=(\d+)', frozen_log)
    assert counts, 'Frozen run log has no trainable parameters line'
    backbone, heads = map(int, counts[-1])   # the log is appended to, so read the newest line
    assert backbone == 0 and heads > 0, f'Frozen run needs zero trainable backbone parameters and positive trainable heads, got backbone={backbone} heads={heads}'
    print('SMOKE run finished. Confirm the frozen run has backbone=0 and a positive heads count.')
    print('A full run has five times as many epochs per configuration. Nine full runs share the available GPUs.')
    print('Set SMOKE = False, then Save Version with Save & Run All.')

## 8. Prototype scoring (P3) on the last epoch of every finished run

In [ ]:
if not SMOKE:
    done = [(c, s) for c in CONFIGS for s in SEEDS if finished(c, s)]
    todo = [(c, s) for c, s in done if not (RUNS_DIR / run_name(c, s) / f'p3_board_ep{planned_epochs(c):02d}.md').is_file()]
    FAILED_P3 = run_queue([p3_job(c, s) for c, s in todo])

## 9. Compare with the control
Saved as `p1f_compare.md` and `p1f_runs.csv`, plus the P3 files `p1f_p3_runs.csv` and `p1f_p3_mean.csv`.

In [ ]:
from cdccp.catalogue import load_catalogue
from cdccp.config import load_config
from cdccp.data import load_split

if not SMOKE:
    cat = load_catalogue(CODE_DIR / load_config()['paths']['catalogue_dev'])
    rows, missing = [], []
    for c in CONFIGS:
        for s in SEEDS:
            run_dir = RUNS_DIR / run_name(c, s)
            if not finished(c, s):
                missing.append(run_name(c, s))
                continue
            h = pd.read_csv(run_dir / 'history.csv')
            sc = run_scores(h)
            with np.load(run_dir / f"val_{sc['pick_epoch']:02d}.npz") as z:
                seen = {k[len('seen_'):]: z[k] for k in z.files if k.startswith('seen_')}
                unseen = {k[len('unseen_'):]: z[k] for k in z.files if k.startswith('unseen_')}
            pb = pair_breakdown(seen, unseen, cat, tol=TOL)
            shortcut = float(h.set_index('epoch').loc[sc['pick_epoch'], 'unseen_shortcut_rate'])
            rows.append({'arm': c, 'seed': s, **sc, **pb, 'shortcut_rate': shortcut})
    run_cols = ['arm', 'seed', 'unseen_disease', 'seen_disease', 'pick_epoch', 'best_epoch', 'best_unseen_disease', 'epochs',
                'crop', 'disease', 'both_heads', 'A_S_raw', 'A_U_raw', 'gamma_keep', 'A_S_keep', 'A_U_keep', 'keep_disease_right', 'shortcut_rate']
    per_run = pd.DataFrame(rows, columns=run_cols)
    assert all(set(r) == set(run_cols) for r in rows), 'run_scores or pair_breakdown changed its keys: update run_cols'
    split_cfg = load_config(CODE_DIR / 'configs' / f'{CONTROL}.yaml', [f'paths.cache_dir={CACHE_DIR}'])
    fit, val_unseen = load_split(split_cfg, 'train_fit'), load_split(split_cfg, 'val_unseen')
    trivial = 100 * (val_unseen.disease_id == fit.disease_id.mode()[0]).mean()
    cols = ['unseen_disease', 'crop', 'both_heads', 'A_U_raw', 'A_U_keep', 'keep_disease_right', 'shortcut_rate', 'seen_disease', 'A_S_keep']
    means = per_run.groupby('arm')[cols].mean().reindex([c for c in CONFIGS if c in set(per_run.arm)]).round(1)
    lines = ['Per run. unseen_disease is the best full three-epoch mean; seen_disease is its matching-window mean.',
             'All other columns use the single selected centre epoch. Tied best windows use the first centre.',
             per_run[['arm', 'seed', 'pick_epoch', *cols]].round(1).to_string(index=False), '',
             'Mean over seeds:', means.to_string(), '',
             f'trivial: always the most common training disease scores {trivial:.2f} unseen disease Top-1', '',
             'unseen_disease: disease head right on unseen pairs (the decision column). crop: crop head right. both_heads: both right on',
             'the same image. A_U_raw / A_U_keep: unseen pair accuracy with no penalty / with the keep-seen penalty. keep_disease_right:',
             'share of A_U_keep hits whose disease head is also right. shortcut_rate: share of wrong diseases that are a disease the true',
             'crop had in training (lower is better).']
    verdicts = []
    for c in CONFIGS[1:]:
        gone = [run_name(a, s) for a in (CONTROL, c) for s in SEEDS if run_name(a, s) in missing]
        if gone:
            verdicts.append(f'{c}: NO VERDICT, runs missing or unfinished: {gone}')
            continue
        _, diffs, mean, limit, word = compare_arms(per_run, CONTROL, c, SEEDS, z=Z)
        per_seed = ', '.join(f'{s}: {d:+.1f}' for s, d in diffs.items())
        verdicts.append(f'{c}: {word.upper()}: {mean:+.1f} points of unseen disease Top-1 against {CONTROL} (bar {limit:.1f}; per seed {per_seed})')
    text = '\n'.join(lines)
    report = ('```\n' + text + '\n```\n\n' + '\n\n'.join(verdicts) + '\n\nThis compares training recipes on development validation. '
              'The score picks its epoch on the same images, so it is not a significance test and not a test-set estimate.')
    print(text, '', *verdicts, sep='\n')
    per_run.to_csv(RUNS_DIR / 'p1f_runs.csv', index=False)
    (RUNS_DIR / 'p1f_compare.md').write_text('# P1f feature-learning recipes vs control (development validation)\n\n' + report + '\n', encoding='utf-8')
    for p in sorted(RUNS_DIR.glob('*/p3_board_ep*.md')):
        print('\n' + p.parent.name + '\n' + p.read_text(encoding='utf-8'))

    print('Disease verdicts alone do not select a final model. Review seen accuracy and pair diagnostics.')
    p3_rows = []
    for c in CONFIGS:
        for seed in SEEDS:
            board = RUNS_DIR / run_name(c, seed) / f'p3_board_ep{planned_epochs(c):02d}.csv'
            if finished(c, seed) and board.is_file():
                b = pd.read_csv(board)
                assert len(b) == 3 and set(b.scorer) == {'head', 'proto-pooled', 'proto-balanced'}, board
                assert 'unseen disease' in b, f'Old P3 report: remove the stale P3 outputs and rerun section 8: {board}'
                p3_rows.append(b.assign(arm=c, seed=seed))
    if p3_rows:
        p3_runs = pd.concat(p3_rows, ignore_index=True)
        p3_runs.to_csv(RUNS_DIR / 'p1f_p3_runs.csv', index=False)
        complete_arms = [c for c in CONFIGS if p3_runs[p3_runs.arm == c].seed.nunique() == len(SEEDS)]
        p3_means = p3_runs[p3_runs.arm.isin(complete_arms)].groupby(['arm', 'scorer'])[
            ['unseen disease', 'unseen shortcut rate', 'A_S keep', 'A_U keep']].mean()
        p3_means.to_csv(RUNS_DIR / 'p1f_p3_mean.csv')
        print('P3 final-epoch diagnostics. These are not the best-window training verdicts.')
        print(p3_means.round(2).to_string())
    else:
        empty_cols = ['arm', 'seed', 'scorer', 'unseen disease', 'unseen shortcut rate', 'A_S keep', 'A_U keep']
        pd.DataFrame(columns=empty_cols).to_csv(RUNS_DIR / 'p1f_p3_runs.csv', index=False)
        pd.DataFrame(columns=[c for c in empty_cols if c != 'seed']).to_csv(RUNS_DIR / 'p1f_p3_mean.csv', index=False)
        print('No completed P3 boards available. Read the P3 failure logs before drawing conclusions.')

## 10. Curves

In [ ]:
if not SMOKE:
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
    colours = dict(zip(CONFIGS, ['tab:blue', 'tab:orange', 'tab:green', 'tab:red']))
    for c in CONFIGS:
        for s in SEEDS:
            path = RUNS_DIR / run_name(c, s) / 'history.csv'
            if not path.is_file():
                continue
            h = pd.read_csv(path)
            kw = dict(color=colours[c], marker='.', label=f'{c} s{s}')
            axes[0].plot(h.epoch, h.unseen_disease_top1, **kw)
            axes[1].plot(h.epoch, h.unseen_shortcut_rate, **kw)
            axes[2].plot(h.epoch, h.seen_disease_top1, **kw)
    titles = ['Unseen disease Top-1', 'Unseen shortcut rate (lower is better)', 'Seen disease Top-1']
    for ax, t in zip(axes, titles):
        ax.set_title(t); ax.set_xlabel('epoch'); ax.grid(alpha=.3); ax.legend(fontsize=6)
    plt.tight_layout(); plt.show()